# Agent-eval reliability, failure modes & benchmark bleed

A look at the **full** set of agentic-eval logs pulled from Braintrust via BTQL
(`data/full.json`, all 1,781 root task spans — the UI export caps at 1,000 rows,
so this is pulled through the API). Each row is one agent rollout scored by an
LLM judge with a binary `task_success`.

The run is described by three knobs:

| dimension | values |
|---|---|
| **harness** | how the agent is scaffolded (`claude_code`, `tool_calling`, `smolagents_code`, `openai_solo`, `tool_calling_with_shortlisting`) |
| **model** | the underlying LLM (`gpt-4.1`, `claude-opus-4-5`, `gemini-3-pro-preview`, `kimi-k2.5`, `deepseek-v3.2`, `gpt-5.2`) |
| **benchmark** | the task suite (`swebench`, `appworld`, `tau2_{retail,airline,telecom}`, `browsecompplus`) |

This notebook extends the basic "who scores highest" view with three questions:

1. **Reliability / uncertainty** — which *configurations* (harness × model) are
   both **successful** *and* **predictable**? Point estimates are shown with
   Wilson confidence intervals and cross-task standard deviations overlaid.
2. **Failure-mode indexing** — when a run fails, *how* does it fail, and do the
   harnesses fail **differently**?
3. **Benchmark bleed** — some suites (notably SWE-bench and the τ²-bench family)
   are public, static, and very likely sit in these models' training data. We
   flag the high-risk suites, exclude them, and check whether the reliability
   conclusions survive.


## 0 · Setup & tidy frame

In [ ]:
import json
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patheffects as pe
import seaborn as sns

warnings.filterwarnings("ignore")
# Braintrust brand theme + reusable chart templates (scripts/braintrust_viz.py).
# This one call registers the Braintrust fonts (bt-sans), applies
# assets/braintrust.mplstyle, and pulls in the palette helpers (rate_cmap,
# diverging_cmap, categorical, harness_colors) plus label/chart builders
# (titled, fig_title, repel_labels, hbar, rate_heatmap, quadrant). The theme is
# bigger-typed, brand-colored, and colorblind-safe (no red/green), per
# https://luiscruz.github.io/2021/03/01/effective-visualizations.html
sys.path.insert(0, "scripts")
from braintrust_viz import *  # noqa: F401,F403  (theme, palettes, builders, mticker)
use_braintrust_theme()


pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Pull straight from Braintrust, cached to disk. First run (or after you delete the
# file) runs the BTQL below for all root task spans and writes data/full.json; every
# run after reads that local snapshot — so the analysis stays offline, fast, and
# reproducible with no separate "export the JSON" step. bt_helpers is the helper in
# scripts/ (paginated, retrying BTQL + JSON cache); a refresh needs BRAINTRUST_API_KEY
# in .env. Run this notebook from the repo root so "scripts/" and "data/" resolve.
sys.path.insert(0, "scripts")
import bt_helpers as bt

DATA = Path("data/full.json")
BTQL = (
    "select: metadata.benchmark AS benchmark, metadata.harness AS harness, "
    "metadata.model AS model, metadata.session_id AS session_id, "
    "metadata.total_tokens AS total_tokens, metadata.num_llm_calls AS num_llm_calls, "
    "metadata.has_errors AS has_errors, metadata.tool_error_count AS tool_error_count, "
    "metadata.error_span_count AS error_span_count, metadata.judge_model AS judge_model, "
    "metadata.judge_confidence AS judge_confidence, metadata.judge_reasoning AS judge_reasoning, "
    "metadata.judge_reliability AS judge_reliability, scores.task_success AS task_success, "
    "span_attributes.name AS task, duration, id "
    "| from: project_logs('6da0ad7f-d092-4d04-95c5-a2ae182883ec') "
    "| filter: is_root = true"
)
raw = bt.cached_pull(BTQL, str(DATA))   # delete data/full.json to refresh from the API
print(f"{len(raw)} rows")


In [ ]:
def norm_model(m: str | None) -> str | None:
    """Collapse provider/casing variants -> a clean model-family label.

    e.g. openai/Azure/gpt-4.1 and Azure/gpt-4.1 -> gpt-4.1;
    gcp/gemini-3-pro-preview -> gemini-3-pro. Strip the provider path, lowercase,
    drop a trailing -preview and date stamp so the ~10 logged strings collapse to
    the 6 underlying models.
    """
    if not m:
        return None
    base = m.split("/")[-1].lower()
    base = re.sub(r"-preview$", "", base)
    base = re.sub(r"-\d{4}-\d{2}-\d{2}$", "", base)  # gpt-5.2-2025-12-11 -> gpt-5.2
    return base


def flatten(r: dict) -> dict:
    # data/full.json is already flat (BTQL select aliased each metadata.* field
    # to a top-level key); we just normalise types and the model label.
    dur = r.get("duration")  # BTQL `duration` is in SECONDS already
    return {
        "harness": r.get("harness"),
        "model": norm_model(r.get("model")),
        "benchmark": r.get("benchmark"),
        "success": r.get("task_success"),
        "has_errors": bool(r.get("has_errors")),
        "tool_error_count": r.get("tool_error_count") or 0,
        "error_span_count": r.get("error_span_count") or 0,
        "num_llm_calls": r.get("num_llm_calls") or 0,
        "total_tokens": r.get("total_tokens") or 0,
        "duration_s": dur if dur is not None else np.nan,
        "judge_model": r.get("judge_model"),
        "judge_confidence": r.get("judge_confidence"),
        "judge_reliability": r.get("judge_reliability"),
        "judge_reasoning": r.get("judge_reasoning") or "",
        "session_id": r.get("session_id"),
    }


df = pd.DataFrame(flatten(r) for r in raw)
# One row has a null task_success (judge never scored it) -> drop before casting.
df = df.dropna(subset=["success"]).reset_index(drop=True)
df["success"] = df["success"].astype(int)
df["config"] = df["harness"] + " · " + df["model"]
print(df.shape)
df.head(3)


In [ ]:
# Sanity: coverage of the design grid
print("models   :", sorted(df.model.unique()))
print("harnesses:", sorted(df.harness.unique()))
print("benchmarks:", sorted(df.benchmark.unique()))
print("\noverall success rate: {:.1%}".format(df.success.mean()))
df.groupby("harness").success.agg(["mean", "count"]).sort_values("mean", ascending=False)


In [ ]:
def wilson(k, n, z=1.96):
    """Wilson score interval for a binomial proportion. Returns (lo, p, hi, half)."""
    if n == 0:
        return (np.nan, np.nan, np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = (z / denom) * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return (center - margin, p, center + margin, margin)


def rate_table(frame, group):
    """Per-group success rate with Wilson 95% CI."""
    rows = []
    for key, g in frame.groupby(group):
        k, n = int(g.success.sum()), len(g)
        lo, p, hi, half = wilson(k, n)
        rows.append({**({group: key} if isinstance(key, str) else dict(zip(group, key))),
                     "n": n, "success_rate": p, "ci_lo": lo, "ci_hi": hi, "ci_half": half})
    return pd.DataFrame(rows)


def config_reliability(frame, min_cell=5, min_total=10):
    """Per-config success, computed the benchmark-fair way.

    Pooling all of a config's rollouts into one rate (a *micro*-average) is
    confounded: each config ran a different mix / number of tasks per suite, so a
    config that happened to run more of an easy benchmark scores higher for free
    (Simpson's paradox). We therefore also report the *macro*-average -- the
    unweighted mean of the config's per-benchmark rates -- which gives every
    suite equal weight regardless of task counts. ``mix_gap = micro - macro`` is
    how much the pooled view flatters (+) or understates (-) a config purely from
    its benchmark mix. ``cross_task_std`` and ``n_benchmarks`` come from the same
    per-(config, benchmark) cells, so the quadrant in 1b is internally consistent.
    """
    micro = rate_table(frame, "config").rename(columns={"success_rate": "micro"})
    cells = rate_table(frame, ["config", "benchmark"])
    cells = cells[cells.n >= min_cell]
    macro = (cells.groupby("config")
             .agg(macro=("success_rate", "mean"),
                  cross_task_std=("success_rate", "std"),
                  n_benchmarks=("benchmark", "nunique")).reset_index())
    out = micro.merge(macro, on="config", how="left")
    out["cross_task_std"] = out["cross_task_std"].fillna(0)
    out["mix_gap"] = out["micro"] - out["macro"]
    # Floor score = Wilson lower bound on the pooled rate: the conservative
    # "you can count on at least this much" number. It folds success AND our
    # certainty about it into one figure -- high mean but small n (wide CI)
    # is penalised, so a high floor means successful *and* predictable.
    out["floor"] = out["ci_lo"]
    return out[out.n >= min_total].reset_index(drop=True)


### 📐 Stats key — what every band, bar, and dot is computed from

Every plot below reuses the same handful of statistics.(`k` = successes, `n` = tasks, `z` = 1.96 for 95%).

- **Success rate** $p = k/n$ — the fraction of tasks the LLM judge scored 1.
- **Wilson 95% CI** (the error bars / `±` half-widths) — a binomial interval that,
  unlike the textbook normal one, stays inside [0, 1] and widens correctly for
  small `n`. Center and half-width:
  $$\text{center}=\frac{p + z^2/2n}{1+z^2/n},\qquad
    \text{half}=\frac{z}{1+z^2/n}\sqrt{\tfrac{p(1-p)}{n}+\tfrac{z^2}{4n^2}}$$
  Wide bars = *few samples*, not necessarily an erratic agent. `floor` = the CI's
  lower bound (`center − half`): "you can count on at least this much."
- **Micro vs macro rate.** *Micro* = pool every rollout into one $k/n$ (flattered by
  an easy task mix). *Macro* = mean of the per-benchmark rates,
  $\frac{1}{B}\sum_b p_b$, giving each suite equal weight — the Simpson's-paradox-safe
  number. `mix_gap = micro − macro` is how much pooling flatters a config.
- **Cross-task std** (the y-axis in 1b) — the spread of a config's per-benchmark
  rates, $\sqrt{\tfrac{1}{B-1}\sum_b (p_b-\bar p)^2}$. Low = consistent across suites.
- **η²** (1d) — share of total success variance explained by one factor:
  $SS_\text{between}/SS_\text{total}$. **Incremental R²** (1h) — the extra variance a
  factor adds *on top of the other two* in the regression, $R^2_\text{full}-R^2_\text{without}$.
- **Box plots** (2b) — box = inter-quartile range (25th–75th pct), line = median,
  whiskers = 1.5×IQR; outliers hidden so the bulk of the distribution is readable.

## 1 · Reliability & uncertainty

A configuration is only useful in production if it is **both** high-scoring
**and** dependable. We separate three things that a single "average success"
number quietly blends together:

- **Statistical uncertainty** — how sure are we of the success *rate* given the
  sample size? Captured by the **Wilson 95% CI**.
- **Benchmark-mix confound** — a pooled average over all of a config's rollouts
  (a *micro*-average) is **not comparable across configs**, because each one ran a
  different mix and number of tasks per suite. A config that happened to run more
  of an easy benchmark scores higher for free — *Simpson's paradox*. We neutralise
  it by holding the benchmark constant: compute each config's rate **per
  benchmark**, then take the **unweighted mean across suites** (a *macro*-average),
  giving every suite equal weight regardless of task counts.
- **Behavioural consistency** — does the config hold up *across* task suites, or
  is a good average hiding a suite where it falls apart? Captured by the
  **standard deviation of the per-benchmark success rates** (the same cells the
  macro-average is built from).

The sweet spot is high **macro** mean, tight CI, low cross-task spread. Even the
macro-average is only strictly comparable between configs that ran the *same set*
of suites — so coverage (`n_benchmarks`) is shown alongside to flag when it isn't.


### 1·0 · Coverage — the design is unbalanced (read this first)

Before any ranking: **not every config ran every benchmark**, and the gaps are
large enough to invalidate naive cross-config comparisons. The matrix below is the
task count per (config, benchmark); blank = never run, grey = run but thin (<5
tasks, too few to score). Two facts to carry through the rest of §1:

- **Only a handful of configs span all 6 suites.** A config's macro-average over 1–2
  suites is *not* comparable to one computed over 6 — they are measuring different
  (and differently hard) things.
- **`browsecompplus` was run by `claude_code` only**, and
  `tool_calling_with_shortlisting` essentially only ran `appworld`. So any
  cross-*harness* number that pools over suites is partly comparing *which suites a
  harness happened to run*, not the harness itself.

The honest fixes follow: §1e and §1h restrict every comparison to a **shared
benchmark set** or adjust for benchmark as a covariate, and the rankings below are
read with coverage (under-covered configs are greyed) firmly in mind.

In [ ]:
cov = (df.groupby(["config", "benchmark"]).size()
       .unstack("benchmark").reindex(columns=sorted(df.benchmark.unique())))
cov = cov.loc[cov.sum(axis=1).sort_values(ascending=False).index]  # busiest configs on top
n_suites = (cov >= 5).sum(axis=1)

fig, ax = plt.subplots(figsize=(10.5, 0.55 * len(cov) + 2.5))
mask = cov.isna() | (cov < 5)               # blank/thin cells -> masked
sns.heatmap(cov, mask=mask, annot=cov.fillna(0).astype(int), fmt="d",
            cmap=rate_cmap(), linewidths=2.5, linecolor="white", vmin=0,
            cbar_kws={"label": "tasks run", "shrink": .6}, ax=ax,
            annot_kws={"fontsize": 11.5, "fontweight": "semibold"})
ax.set_facecolor(NEUTRAL["panel"])           # masked (blank/thin) cells show grey
# overlay thin-but-nonzero counts in grey so they're visible as "ran but <5"
for yi, cfg in enumerate(cov.index):
    for xi, bm in enumerate(cov.columns):
        v = cov.loc[cfg, bm]
        if pd.notna(v) and v < 5:
            ax.text(xi + 0.5, yi + 0.5, int(v), ha="center", va="center",
                    fontsize=16, color=NEUTRAL["faint"])
for yi, cfg in enumerate(cov.index):         # suite-coverage count on the right
    ax.text(len(cov.columns) + 0.15, yi + 0.5, f"{n_suites[cfg]}/6", va="center",
            ha="left", fontsize=16, weight="bold",
            color=BRAND["blue"] if n_suites[cfg] >= 5 else BRAND["orange"])
titled(ax, "Config × benchmark coverage", "cell = tasks run · grey = <5 (too thin) · right = suites covered")
ax.set_xlabel(""); ax.set_ylabel(""); ax.tick_params(length=0); ax.grid(False)
# Short suite names so the column labels read horizontally without overlapping.
SHORT_BM = lambda b: b.replace("tau2_", "").replace("browsecompplus", "browse")
ax.set_xticklabels([SHORT_BM(b) for b in cov.columns], rotation=0)
name_fig(fig, "01_coverage")
plt.tight_layout(); plt.show()
print(f"configs covering all 6 suites: {(n_suites == 6).sum()} of {len(n_suites)}")


### 1a · Pooled vs benchmark-balanced success per configuration

In [ ]:
MIN_N = 10  # configs with fewer rollouts are too noisy to rank
relc = (config_reliability(df, min_cell=5, min_total=MIN_N)
        .dropna(subset=["macro"]).sort_values("macro").reset_index(drop=True))

COMPARABLE_BM = 3  # configs spanning fewer suites aren't fairly cross-comparable
fig, ax = plt.subplots(figsize=(12, 0.74 * len(relc) + 2.5))
# Bars = pooled (micro) rate with Wilson CI -> statistical uncertainty. Under-covered
# configs (<3 suites) are greyed rather than coloured — they are not cross-comparable.
err = np.vstack([relc.micro - relc.ci_lo, relc.ci_hi - relc.micro])
rate_col = rate_cmap()(relc.macro)
thin_mask = relc.n_benchmarks < COMPARABLE_BM
bar_col = [NEUTRAL["grid"] if t else rate_col[i] for i, t in enumerate(thin_mask)]
ax.barh(range(len(relc)), relc.micro, xerr=err, color=bar_col,
        error_kw=dict(ecolor=NEUTRAL["muted"], capsize=3.5, lw=1.3), height=0.72, zorder=3)
# Diamonds = benchmark-balanced (macro) rate -> the mix-corrected estimate.
ax.scatter(relc.macro, range(len(relc)), marker="D", s=80, color=BRAND["orange"],
           edgecolor="white", linewidth=1.2, zorder=5)
# n folded into the y-axis label so there's no separate, noisy right-hand column.
ax.set_yticks(range(len(relc)))
ax.set_yticklabels([f"{c}   (n={n})" for c, n in zip(relc.config, relc.n)], fontsize=16)
for y, r in enumerate(relc.micro):           # one value label per bar, at its end
    ax.text(r + 0.012, y, f"{r:.0%}", va="center", ha="left",
            fontsize=16, weight="bold", color=NEUTRAL["body"])
ax.set_xlim(0, 1.16)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel("task success rate")
ax.set_ylabel("")
ax.tick_params(axis="x", labelsize=12)
titled(ax, "Pooled bars vs benchmark-balanced diamonds",
       "bar = pooled rate (±Wilson 95%) · ♦ = benchmark-balanced (macro) · "
       "grey bar = <3 suites, not cross-comparable", pad=24)
ax.margins(y=0.01)
sns.despine(ax=ax, left=True)
ax.tick_params(left=False)
name_fig(fig, "02_1a")
plt.tight_layout()
plt.show()

relc.sort_values("floor", ascending=False)[
    ["config", "n", "n_benchmarks", "micro", "macro", "floor", "mix_gap", "cross_task_std"]].round(3)


**Bars** are the pooled (micro) rate with the Wilson 95% CI — wide bars are
*unreliable estimates* (small `n`), not necessarily unreliable agents.
**Diamonds** are the benchmark-balanced (macro) rate. Where a diamond sits well to
the **left** of its bar, the pooled headline is **flattered by an easy benchmark
mix** (`mix_gap > 0`); to the **right**, the pooled number *understates* the
config. **Grey bars** ran fewer than 3 suites, so their macro-average isn't
comparable to a full-coverage config's — treat them as "insufficient evidence to
rank." Configs whose Wilson CIs overlap heavily are statistically indistinguishable.

The table is **sorted by `floor`** — the Wilson *lower* bound on the pooled rate.
Read it as "whatever else is true, this config delivers at least this much,"
which is the single number that rewards being **both** high-scoring **and**
well-sampled/predictable: a flashy mean on thin data has a low floor and sinks in
the ranking.

**Coverage gate:** **grey bars** ran fewer than 3 suites, so their position on this
axis is *not* comparable to the full-coverage configs — they may simply have run an
easier (or harder) subset.
Treat them as "insufficient evidence to rank," not as genuinely better/worse, and
rely on the other plots for cross-config comparison.

### 1b · Success vs. predictability — the reliability quadrant

In [ ]:
# Mean (macro) and spread (cross-task std) come from the SAME per-(config,
# benchmark) cells, so the quadrant can't be gamed by benchmark mix.
rel = relc[relc.n_benchmarks >= 2].copy()  # need >=2 suites for a cross-task std

fig, ax = plt.subplots(figsize=(14, 9.5))
xm = rel.macro.median(); ym = rel.cross_task_std.median()
ax.axvline(xm, ls="--", c=NEUTRAL["grid"], lw=1.2, zorder=0)
ax.axhline(ym, ls="--", c=NEUTRAL["grid"], lw=1.2, zorder=0)
# Smaller bubbles + extra margin leave room for labels in the tight bottom-right cluster.
sns.scatterplot(data=rel, x="macro", y="cross_task_std", size="n",
                hue="macro", palette=rate_cmap(), sizes=(60, 300),
                edgecolor="white", linewidth=1.3, legend=False, ax=ax, zorder=3)
# Aggressive de-collision; abbreviated labels (cc · opus-4.5 …) keep the cluster legible.
repel_labels(ax, rel.macro, rel.cross_task_std, [abbrev_config(c) for c in rel.config],
             expand=(1.8, 2.2), force_text=(1.4, 1.8), force_static=(0.8, 1.1))
ax.text(0.985, 0.04, "RELIABLE\nhigh score · consistent", transform=ax.transAxes,
        ha="right", va="bottom", color=BRAND["blue"], fontsize=16, weight="bold", linespacing=1.3)
ax.text(0.015, 0.97, "ERRATIC\nlow score · inconsistent", transform=ax.transAxes,
        ha="left", va="top", color=BRAND["orange"], fontsize=16, weight="bold", linespacing=1.3)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel("benchmark-balanced (macro) success  →  higher is better")
ax.set_ylabel("cross-task std of success  →  lower is more predictable")
titled(ax, "Reliability quadrant — dependable configs sit bottom-right",
       "x = macro rate (mean of per-suite rates) · y = std of those per-suite rates · "
       "bubble = sessions (see Stats key)")
ax.margins(0.18)
name_fig(fig, "03_1b")
plt.tight_layout(); plt.show()
rel.sort_values(["macro", "cross_task_std"], ascending=[False, True])[
    ["config", "n", "n_benchmarks", "micro", "macro", "floor", "mix_gap", "cross_task_std", "ci_half"]].round(3)


**Read it as:** bottom-right = the dependable workhorses (score high,
behave the same on every suite). Top-left = avoid. A config high on the x-axis
but also high on the y-axis is a *gambler*: great on average, but with a suite
where it collapses. Because the x-axis is the macro (benchmark-balanced) rate and
the y-axis is the spread of the *same* per-suite cells, a config can't buy its way
rightward by running more easy tasks — the Simpson's-paradox loophole from §1a is
closed here.

### 1c · Harness × model success heatmap (mean ± Wilson half-width)

In [ ]:
hm = rate_table(df, ["harness", "model"])
hm = hm[hm.n >= 5]
pivot = hm.pivot(index="harness", columns="model", values="success_rate")

# Colour alone carries the rate (read off the colourbar) — no per-cell numbers, so the
# grid stays scannable. Exact rates, CIs and n live in §1a's table.
fig, ax = plt.subplots(figsize=(12, 5.8))
sns.heatmap(pivot, annot=False, cmap=rate_cmap(), vmin=0, vmax=1,
            linewidths=3, linecolor="white",
            cbar_kws={"label": "success rate", "shrink": .85}, ax=ax)
titled(ax, "Success by harness × model", "colour = pooled success rate (n ≥ 5) · exact rates, CI & n in §1a")
ax.set_xlabel(""); ax.set_ylabel("")
ax.tick_params(length=0); ax.grid(False)
plt.setp(ax.get_xticklabels(), rotation=0)
plt.setp(ax.get_yticklabels(), rotation=0)
name_fig(fig, "04_1c")
plt.tight_layout(); plt.show()


The `±` term is the statistical noise floor for each cell. Where the
half-width is large, the cell is under-sampled — treat colour differences with
caution. Each cell also still **pools across benchmarks**, so it inherits the same
mix confound as §1a's bars — use it to spot broad patterns, and lean on the
per-benchmark macro rates in §1a/§1b for fair ranking.

### 1d · What moves the needle — harness vs model vs benchmark

The heatmap hints that *how* you scaffold the agent matters more than *which*
model you drop in. We make that quantitative with a one-way **η²** (eta-squared):
for each factor, the share of the total variance in `success` explained by knowing
only that factor's value. Higher = that knob swings the outcome more.

Caveat: the design is **unbalanced** (not every harness ran every model×benchmark),
so these one-way effects are partly confounded with each other and **do not sum to
100%**. They are a directional read on *which knob dominates*, not a clean variance
decomposition — read the size of the gap between bars, not their absolute values.

In [ ]:
def eta_sq(frame, factor):
    """One-way eta-squared: SS_between / SS_total for `success` grouped by factor."""
    grand = frame.success.mean()
    ss_tot = ((frame.success - grand) ** 2).sum()
    if ss_tot == 0:
        return 0.0
    ss_bet = sum(len(g) * (g.success.mean() - grand) ** 2
                 for _, g in frame.groupby(factor))
    return ss_bet / ss_tot


factors = ["harness", "model", "benchmark", "config"]
eta = pd.DataFrame({"factor": factors,
                    "eta_sq": [eta_sq(df, f) for f in factors]}).sort_values("eta_sq")

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#3b82f6" if f != "config" else "#9eb6ff" for f in eta.factor]
ax.barh(eta.factor, eta.eta_sq, color=colors, height=0.66, zorder=3)
for y, v in enumerate(eta.eta_sq):
    ax.text(v + 0.004, y, f"{v:.1%}", va="center", fontsize=16, weight="bold",
            color=NEUTRAL["body"])
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlim(0, max(eta.eta_sq) * 1.18)
ax.set_xlabel("share of success variance explained (one-way η²)")
titled(ax, "Which knob moves task success most?",
       "config = harness × model jointly · unbalanced design, bars don't sum to 100%")
sns.despine(ax=ax, left=True); ax.tick_params(left=False)
name_fig(fig, "05_1d")
plt.tight_layout(); plt.show()
eta.set_index("factor").round(4)


If **harness** outranks **model**, the scaffolding is the bigger lever —
swapping in a stronger LLM buys less than fixing how it is driven. `config`
(harness × model jointly) sits highest by construction; the gap between it and the
single factors is the slice explained by the *interaction* — i.e. the right model
needs the right harness.

### 1e · Harness swing for a fixed model — on a *shared* benchmark set

Hold the **model** constant and watch success move as the harness changes. But per
§1·0, a model's harnesses ran *different* suites, so the earlier "macro over each
harness's own suites" version mixed the harness effect with coverage. Here we close
that hole: for each model we take only the benchmarks that **every** one of its
(qualifying) harnesses ran (≥5 tasks each), and score all harnesses **on that shared
set only**. Each panel below is one model; each bar is a harness's pooled success on
the shared suites (±Wilson 95%). The **Δ** in each title is the best−worst gap —
the apples-to-apples swing from scaffolding — and each panel **names the shared
suites underneath**, because *which* suites (and how many) back the swing is exactly
what tells you how far to trust it. Note that for Claude, Gemini, and DeepSeek the
only common ground is **appworld** — so their report card is really an appworld-only
read — whereas gpt-4.1 spans two τ²-bench suites.

In [ ]:
MIN_CELL, MIN_TOT = 5, 10
# Short, unambiguous suite labels for the panel titles (the tau2_* family drops its
# prefix — retail/airline/telecom are only ever tau2, so no information is lost).
BENCH_ABBR = {"swebench": "swebench", "appworld": "appworld",
              "browsecompplus": "browse", "tau2_airline": "airline",
              "tau2_retail": "retail", "tau2_telecom": "telecom"}
cells_mhb = rate_table(df, ["model", "harness", "benchmark"])
cells_mhb = cells_mhb[cells_mhb.n >= MIN_CELL]
tot = cells_mhb.groupby(["model", "harness"]).n.sum()
qual = {mh for mh, v in tot.items() if v >= MIN_TOT}  # harnesses with enough data

records, shared_bms = [], {}
for model in sorted(df.model.unique()):
    hs = sorted(h for (m, h) in qual if m == model)
    if len(hs) < 2:
        continue
    bm_sets = [set(cells_mhb[(cells_mhb.model == model) & (cells_mhb.harness == h)].benchmark)
               for h in hs]
    common = set.intersection(*bm_sets)            # suites EVERY harness of this model ran
    if not common:
        continue
    # Name the shared suites (not just count them): a swing measured on a single
    # suite is much weaker evidence than one measured across several, and which
    # suite it is matters (e.g. appworld is hard for Claude). Surface both.
    shared_bms[model] = ", ".join(BENCH_ABBR.get(b, b) for b in sorted(common))
    sub = df[(df.model == model) & (df.benchmark.isin(common))]
    for h in hs:
        gh = sub[sub.harness == h]
        lo, p, hi, _ = wilson(int(gh.success.sum()), len(gh))
        records.append({"model": model, "harness": h, "rate": p,
                        "ci_lo": lo, "ci_hi": hi, "n": len(gh)})
ce = pd.DataFrame(records)
swing = (ce.groupby("model").rate.agg(["min", "max"])
         .assign(swing=lambda d: d["max"] - d["min"],
                 shared=lambda d: d.index.map(shared_bms)).sort_values("swing"))

hh = sorted(df.harness.unique())
hpal = harness_colors(hh)   # same color per harness in every chart of the notebook
models_ord = swing.sort_values("swing", ascending=False).index.tolist()

# One standalone figure per model (its own report card) instead of a dense facet grid.
for model in models_ord:
    sub = ce[ce.model == model].sort_values("rate").reset_index(drop=True)
    err = np.vstack([sub.rate - sub.ci_lo, sub.ci_hi - sub.rate])
    fig, ax = plt.subplots(figsize=(9.5, 0.72 * len(sub) + 2.2))
    ax.barh(range(len(sub)), sub.rate, xerr=err, height=0.6,
            color=[hpal[h] for h in sub.harness],
            error_kw=dict(ecolor=NEUTRAL["muted"], capsize=3.5, lw=1.3), zorder=3)
    for y, s in sub.iterrows():
        # label past the whisker tip so it reads as the point estimate, not a tag on it
        ax.text(max(s.ci_hi, s.rate) + 0.02, y, f"{s.rate:.0%}", va="center", ha="left",
                fontsize=16, weight="bold", color=NEUTRAL["body"])
    ax.set_yticks(range(len(sub))); ax.set_yticklabels(sub.harness, fontsize=16)
    ax.set_xlim(0, 1.12); ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.set_xlabel("task success rate")
    # Title names the shared suite(s) the swing is measured on — a Δ on appworld-only
    # reads differently from one spanning several suites.
    titled(ax, f"{model}   ·   Δ {swing.loc[model, 'swing']:.0%} best−worst harness",
           f"success on shared suites: {swing.loc[model, 'shared']}  (±Wilson 95%)", pad=24)
    sns.despine(ax=ax, left=True); ax.tick_params(left=False)
    name_fig(fig, f"06_1e_{model}")
    plt.tight_layout(); plt.show()
swing[["min", "max", "swing"]].round(3).assign(shared=swing["shared"])


Read each panel as a model's **report card**: how well does it do under each
harness, scored on the *same* tasks? The bigger the spread between the top and
bottom bar (the **Δ** in the title), the more that model's success is dictated by
its scaffolding rather than its weights. Even on the shared suites the spreads stay
large — so the harness effect is **not** a coverage artefact. Read the **shared-suites
label under each title** before trusting a swing: a panel whose only shared suite is
`appworld` is an appworld-only verdict (narrower evidence than gpt-4.1's two-suite
span), even though the Δ can look just as dramatic. Bars whose Wilson whiskers don't
overlap differ beyond sampling noise.

### 1f · Where do the 'gamblers' collapse?

§1b flagged configs with a high average but a high **cross-task std** — strong on
some suites, brittle on others. The cross-task std is a single number; this heatmap
is the receipt. Each cell is a config's success on one benchmark (greyed where that
config ran <5 tasks on the suite, too thin to read). Scan a row: an all-deep-blue row is
a genuine generalist; a row with a pale cell is a gambler whose headline average hid
a suite where it falls apart.

In [ ]:
cb = rate_table(df, ["config", "benchmark"])
cb_n = cb.pivot(index="config", columns="benchmark", values="n")
cb_rate = cb.pivot(index="config", columns="benchmark", values="success_rate")
# Drop configs with no cell that clears the n>=5 bar (nothing to show but an empty row).
keep_rows = ((cb_n >= 5).sum(axis=1) > 0)
cb_rate, cb_n = cb_rate[keep_rows], cb_n[keep_rows]
# Order rows by overall macro (mean of available per-benchmark rates), best on top.
order_idx = cb_rate.mean(axis=1).sort_values(ascending=False).index
cb_rate, cb_n = cb_rate.loc[order_idx], cb_n.loc[order_idx]
# Mask thin cells (n < 5) so they read as "not enough data", not "scored 0".
mask = (cb_n < 5) | cb_rate.isna()

# Colour carries the rate (read the colourbar); dropping per-cell numbers keeps a tall
# matrix scannable. A pale cell in an otherwise deep-blue row is the brittle suite.
fig, ax = plt.subplots(figsize=(10.5, 0.55 * len(cb_rate) + 2.5))
sns.heatmap(cb_rate, mask=mask, annot=False, cmap=rate_cmap(), vmin=0, vmax=1,
            linewidths=3, linecolor="white", cbar_kws={"label": "success rate", "shrink": .7},
            ax=ax)
ax.set_facecolor(NEUTRAL["panel"])  # masked (thin) cells show as grey
titled(ax, "Config × benchmark success — spotting the gamblers",
       "colour = success rate · grey = <5 tasks (too thin) · a pale cell in a deep-blue row = a brittle suite")
ax.set_xlabel(""); ax.set_ylabel("")
ax.tick_params(length=0); ax.grid(False)
# Short suite names so the column labels read horizontally without overlapping.
ax.set_xticklabels([b.replace("tau2_", "").replace("browsecompplus", "browse")
                    for b in cb_rate.columns], rotation=0)
name_fig(fig, "07_1f")
plt.tight_layout(); plt.show()


Read with §1b: a config can earn a high macro average yet still own a
single pale cell — that suite is where it will burn you in production. Conversely an
all-deep-blue row with tight cells (e.g. the top `claude_code` configs) is what
"reliable" actually looks like: high *and* even across task types.

### 1g · Where the harness actually changes the outcome

§1e held the model fixed but averaged over suites. This fixes **both the model and
the benchmark** — the cleanest cut at the harness effect, difficulty confound and all.

**The honest filter:** a harness comparison only *exists* where **≥2 harnesses ran the
same (model, benchmark)** — anywhere else there is literally nothing to compare, so we
drop it. That alone removes a lot of the grid (e.g. `browsecompplus` disappears
entirely — only `claude_code` ever ran it), leaving just the cells that carry a real
signal. Each surviving panel is one (benchmark, model); bars are the harnesses that
ran it (±Wilson 95%), and panels are **ranked by Δ** — the success gap between the
best and worst harness — so the most decisive cases come first.

In [ ]:
cbm = rate_table(df, ["benchmark", "model", "harness"])
cbm = cbm[cbm.n >= 5]
# A (benchmark, model) cell is informative ONLY if >=2 harnesses ran it -- otherwise
# there is no harness comparison to make. Keep just those; rank by the harness swing.
keep = cbm.groupby(["benchmark", "model"]).harness.nunique().loc[lambda s: s >= 2].index
cells = [cbm[(cbm.benchmark == b) & (cbm.model == m)].sort_values("success_rate")
         for (b, m) in keep]
cells.sort(key=lambda g: g.success_rate.max() - g.success_rate.min(), reverse=True)

hh = sorted(df.harness.unique())
hpal = harness_colors(hh)   # same color per harness as every other chart
# One standalone figure per (benchmark, model) head-to-head, ranked by Δ (most decisive
# first). rank prefix keeps the export order stable and readable.
for i, grp in enumerate(cells):
    err = np.vstack([grp.success_rate - grp.ci_lo, grp.ci_hi - grp.success_rate])
    b, m = grp.iloc[0][["benchmark", "model"]]
    delta = grp.success_rate.max() - grp.success_rate.min()
    fig, ax = plt.subplots(figsize=(9.5, 0.72 * len(grp) + 2.2))
    ax.barh(range(len(grp)), grp.success_rate, xerr=err, height=0.6,
            color=[hpal[h] for h in grp.harness],
            error_kw=dict(ecolor=NEUTRAL["muted"], capsize=3.5, lw=1.3), zorder=3)
    for y, (_, s) in enumerate(grp.iterrows()):
        ax.text(max(s.ci_hi, s.success_rate) + 0.02, y, f"{s.success_rate:.0%}", va="center",
                ha="left", fontsize=16, weight="bold", color=NEUTRAL["body"])
    ax.set_yticks(range(len(grp))); ax.set_yticklabels(grp.harness, fontsize=16)
    ax.set_xlim(0, 1.12); ax.set_ylim(-0.6, len(grp) - 0.4)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.set_xlabel("task success rate")
    titled(ax, f"{b} · {m}   ·   Δ {delta:.0%}",
           "harnesses that ran this (model, benchmark) · ±Wilson 95%", pad=24)
    sns.despine(ax=ax, left=True); ax.tick_params(left=False)
    name_fig(fig, f"08_1g_{i+1:02d}_{b}_{m}")
    plt.tight_layout(); plt.show()
print(f"{len(cells)} (benchmark, model) cells have a real harness comparison "
      f"(of {cbm.groupby(['benchmark','model']).ngroups} populated cells)")


Difficulty stripped out, the harness effect doesn't wash away — within a single
suite the *same model* swings by tens of points on scaffolding alone (top-left panels
clear **60 points**). These are the most defensible harness comparisons in the whole
analysis: the two biggest confounds, **model and task difficulty, are both held
fixed**, and only cells with an actual head-to-head are shown.

### 1h · Confound-adjusted harness effect (two-way model)

§1e/§1g control the confounds by *slicing*; this controls them by *modelling*. We
fit a **linear probability model**

`success ~ C(benchmark) + C(model) + C(harness)`

with heteroskedasticity-robust (HC1) standard errors. Because benchmark and model
are in the model, each **harness coefficient is the effect of that harness in
percentage points, holding the suite mix and the model fixed** — exactly the
coverage-adjusted number the raw averages couldn't give. `tool_calling` (the
weakest) is the reference, so every coefficient reads as "points above
`tool_calling`, all else equal."

> **Adjusted ≠ balanced.** This is *not* the benchmark-**balanced** (macro) rate
> from §1a/§1b. There, we average each config's per-suite rates with **equal weight
> per suite**. Here, benchmark is a **covariate** that *partials out* its main
> effect, but every rollout still counts **once** — suites with more tasks pull the
> fit harder; we do not re-weight them to equal size. Both neutralise the same
> coverage confound, by different means: §1a **re-weights**, §1h **adjusts**.

To settle *harness vs model* fairly, we also compute each factor's **incremental
R²** — how much variance it explains when *added on top of the other two*. That is
the adjusted analogue of §1d's one-way η², with the coverage confound removed.

In [ ]:
import statsmodels.formula.api as smf

dfm = df.dropna(subset=["harness", "model", "benchmark"]).copy()
full = smf.ols("success ~ C(benchmark) + C(model) + "
               "C(harness, Treatment(reference='tool_calling'))",
               data=dfm).fit(cov_type="HC1")

# --- harness coefficients (vs tool_calling reference) ---
hp = []
for term in full.params.index:
    if term.startswith("C(harness"):
        name = term.split("[T.")[1].rstrip("]")
        ci = full.conf_int().loc[term]
        hp.append({"harness": name, "coef": full.params[term],
                   "lo": ci[0], "hi": ci[1], "p": full.pvalues[term]})
hp.append({"harness": "tool_calling", "coef": 0.0, "lo": 0.0, "hi": 0.0, "p": np.nan})  # reference
hp = pd.DataFrame(hp).sort_values("coef")

# --- incremental R^2: variance each factor adds on top of the other two ---
def r2(formula):
    return smf.ols(formula, data=dfm).fit().rsquared
r2_full = r2("success ~ C(benchmark) + C(model) + C(harness)")
incr = pd.Series({
    "harness":   r2_full - r2("success ~ C(benchmark) + C(model)"),
    "model":     r2_full - r2("success ~ C(benchmark) + C(harness)"),
    "benchmark": r2_full - r2("success ~ C(model) + C(harness)"),
}).sort_values()

# --- figure 1: harness coefficient forest plot (its own image) ---
fig, axL = plt.subplots(figsize=(11, 5.2))
# effect labels sit in one fixed column to the right of every CI line, so each number
# reads as the point estimate rather than a tag on its whisker tip
label_x = hp.hi.max() + 0.03
for y, (_, r) in enumerate(hp.iterrows()):
    is_ref = r.harness == "tool_calling"
    c = NEUTRAL["faint"] if is_ref else (BRAND["blue"] if r.coef > 0 else BRAND["orange"])
    if not is_ref:
        axL.plot([r.lo, r.hi], [y, y], color=c, lw=2.4, alpha=0.5, zorder=2)
    axL.scatter(r.coef, y, s=150, color=c, edgecolor="white", linewidth=1.4, zorder=3)
    tag = "(ref)" if is_ref else (f"+{r.coef:.0%}" if r.coef >= 0 else f"{r.coef:.0%}")
    axL.text(label_x, y, tag, va="center", ha="left", fontsize=16,
             weight="bold", color=c)
axL.axvline(0, ls="--", c=NEUTRAL["grid"], lw=1.2, zorder=0)
axL.set_xlim(hp.lo.min() - 0.03, label_x + 0.10)
axL.set_yticks(range(len(hp))); axL.set_yticklabels(hp.harness)
axL.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axL.set_xlabel("effect on success vs. tool_calling (pp, adjusted for model + benchmark)")
titled(axL, "Adjusted harness effect", "linear prob. model · HC1 SEs · 95% CI · blue > ref, orange < ref", pad=20)
sns.despine(ax=axL, left=True); axL.tick_params(left=False)
name_fig(fig, "09_1h_forest")
plt.tight_layout(); plt.show()

# --- figure 2: incremental R^2 by factor (its own image) ---
fig, axR = plt.subplots(figsize=(9, 4))
colors = {"harness": "#3b82f6", "model": "#9eb6ff", "benchmark": "#d4d4d8"}
axR.barh(incr.index, incr.values, color=[colors[f] for f in incr.index], height=0.64, zorder=3)
for y, v in enumerate(incr.values):
    axR.text(v + 0.003, y, f"{v:.1%}", va="center", fontsize=16, weight="bold", color=NEUTRAL["body"])
axR.set_xlim(0, max(incr.values) * 1.2)
axR.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axR.set_xlabel("incremental R²  (added on top of the other two)")
titled(axR, "Which knob matters, adjusted?", "coverage confound removed", pad=20)
sns.despine(ax=axR, left=True); axR.tick_params(left=False)
name_fig(fig, "09_1h_incremental-r2")
plt.tight_layout(); plt.show()

print(f"full model R² = {r2_full:.3f}  (n={int(full.nobs)})")
hp.set_index("harness")[["coef", "lo", "hi", "p"]].round(3)


**This is the headline, properly adjusted.** The forest plot ranks harnesses
by their effect on success *with model and benchmark held fixed* — `tool_calling`
is the floor and the code/agentic harnesses sit well above it, with CIs that clear
zero. The incremental-R² chart is the clean version of §1d: even after removing the coverage
confound, **harness adds more incremental R² than model does**, confirming the
scaffolding is the larger lever. (Recall this is benchmark-**adjusted** — benchmark
enters as a covariate so its main effect is partialled out — *not* benchmark-
**balanced** like §1a's macro rate, which equal-weights the suites; the two are
different routes to the same coverage-confound fix. LPM is used for interpretable
percentage-point effects; a logit fit gives the same ordering and significance.)

## 2 · Failure-mode indexing — do harnesses fail differently?

We have several orthogonal error signals per run:

- `tool_error_count` — failed tool/function calls
- `error_span_count` — spans the tracer marked as errored
- `total_tokens` / `num_llm_calls` — proxy for non-convergence / looping
- `judge_reasoning` — free-text explanation from the judge

Among the **failed** runs we bucket each into a dominant failure mode, then ask
whether the mix of modes differs by harness.

> **Coverage caveat (per §1·0):** harnesses ran *different* benchmark mixes, and
> different suites fail in different ways (a `swebench` failure looks nothing like a
> `tau2` one). So a harness's *pooled* failure fingerprint partly reflects **which
> suites it happened to run**, not how the harness itself fails. The pooled views
> (§2a/§2b/§2c) are kept as an overview but flagged as confounded; the honest
> read is the **benchmark-faceted** version beside each, which holds the suite
> fixed and only compares harnesses that actually ran it.


In [ ]:
fails = df[df.success == 0].copy()
print(f"{len(fails)} failed runs ({len(fails)/len(df):.0%} of all)")

tok_hi = df.total_tokens.quantile(0.90)  # global "blow-up" threshold

def classify(r):
    if r.tool_error_count >= 5:
        return "tool-call errors"
    if r.total_tokens >= tok_hi or r.num_llm_calls >= df.num_llm_calls.quantile(0.90):
        return "non-convergence / loop"
    if r.error_span_count > 0:
        return "runtime / span error"
    return "silent wrong answer"

fails["mode"] = fails.apply(classify, axis=1)
fails["mode"].value_counts()


### 2a · Failure-mode mix by harness (composition, not volume)

In [ ]:
order = fails.harness.value_counts().index.tolist()
modes = ["tool-call errors", "non-convergence / loop", "runtime / span error", "silent wrong answer"]
counts = (fails.groupby(["harness", "mode"]).size()
          .unstack("mode").reindex(order)[modes].fillna(0))
mp = counts.div(counts.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(12, 6.5))
bottom = np.zeros(len(mp))
colors = categorical(len(modes))
n_lab = fails.harness.value_counts().reindex(order)
xpos = np.arange(len(mp))
for c, m in zip(colors, modes):
    ax.bar(xpos, mp[m], bottom=bottom, label=m, color=c, width=0.68, zorder=3)
    for x, (frac, b) in enumerate(zip(mp[m].values, bottom)):
        if frac >= 0.08:  # only label visible segments
            ax.text(x, b + frac / 2, f"{frac:.0%}", ha="center", va="center",
                    fontsize=16, color="white", weight="bold")
    bottom += mp[m].values
ax.set_ylabel("share of harness's failures")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xticks(xpos)
ax.set_xticklabels([f"{h}\n(n={n_lab[h]})" for h in mp.index], rotation=0, ha="center")
ax.legend(title="failure mode", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
titled(ax, "How each harness fails", "normalised within harness — bars sum to 100%")
sns.despine(ax=ax)
name_fig(fig, "10_2a")
plt.tight_layout(); plt.show()
mp.round(2)


If harnesses failed the same way, every bar would have identical
stripes. Differences here are the harness-specific fingerprints — e.g. a
code-execution harness leaning on `tool-call errors` vs. a solo prompt leaning
on `silent wrong answer`. **But** (see the caveat above) this pools each harness
over its own suite mix — so part of any difference is just *which benchmarks* the
harness ran. The next chart removes that.

### 2a-2 · Failure-mode mix by harness, **within each benchmark**

In [ ]:
MIN_FAIL = 15  # need this many failures in a (benchmark, harness) cell to show it
fc = fails.groupby(["benchmark", "harness", "mode"]).size().rename("k").reset_index()
cell_tot = fails.groupby(["benchmark", "harness"]).size().rename("tot")
fc = fc.join(cell_tot, on=["benchmark", "harness"])
fc["frac"] = fc.k / fc.tot
# keep (benchmark, harness) cells with enough failures, and benchmarks with >=2 such harnesses
ok_cells = cell_tot[cell_tot >= MIN_FAIL]
ok = ok_cells.reset_index()
bench_keep = ok.groupby("benchmark").harness.nunique()
bench_keep = bench_keep[bench_keep >= 2].index.tolist()

mode_colors = dict(zip(modes, categorical(len(modes))))
# One standalone figure per benchmark, each with its own legend.
for bench in sorted(bench_keep):
    hs = sorted(ok[ok.benchmark == bench].harness)
    sub = fc[(fc.benchmark == bench) & (fc.harness.isin(hs))]
    piv = sub.pivot_table(index="harness", columns="mode", values="frac", fill_value=0).reindex(hs)
    piv = piv.reindex(columns=modes, fill_value=0)
    fig, ax = plt.subplots(figsize=(max(7, 1.7 * len(piv) + 3.5), 6))
    bottom = np.zeros(len(piv))
    xpos = np.arange(len(piv))
    for m in modes:
        ax.bar(xpos, piv[m], bottom=bottom, color=mode_colors[m], width=0.62, zorder=3, label=m)
        for x, (frac, b) in enumerate(zip(piv[m].values, bottom)):
            if frac >= 0.08:
                ax.text(x, b + frac / 2, f"{frac:.0%}", ha="center", va="center",
                        fontsize=16, color="white", weight="bold")
        bottom += piv[m].values
    ax.set_xticks(xpos)
    ax.set_xticklabels([f"{h}\n(n={int(cell_tot[(bench, h)])})" for h in piv.index],
                       fontsize=16, rotation=0)
    ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.set_ylabel("share of harness's failures")
    ax.legend(title="failure mode", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
    titled(ax, f"How each harness fails on {bench}",
           f"failures bucketed by mode · bars sum to 100% · (harness, benchmark) cells with ≥{MIN_FAIL} failures")
    sns.despine(ax=ax)
    name_fig(fig, f"11_2a2_{bench}")
    plt.tight_layout(); plt.show()


Now a harness's fingerprint can be compared *like-for-like*: each panel fixes
the benchmark, so any difference in stripe pattern is the harness, not the task mix.
Where a harness keeps the same fingerprint across panels, that's a genuine
harness-level failure signature; where its stripes change suite-to-suite, the
"fingerprint" from the pooled chart was partly a coverage artefact.

### 2b · Error-signal distributions by harness (failed runs)

In [ ]:
short = {h: h.replace("_", "\n", 1).replace("tool_calling_with_shortlisting", "tool_calling\n+shortlist")
         for h in order}
hpal_b = harness_colors(order)
SUB_BOX = "box = IQR (25th–75th pct) · line = median · whiskers = 1.5×IQR · outliers hidden"

# Two standalone figures — each error signal its own image.
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=fails, x="harness", y="tool_error_count", order=order,
            hue="harness", palette=hpal_b, legend=False, ax=ax,
            showfliers=False, linewidth=1.3, width=0.6)
titled(ax, "Tool-call errors per failed run", SUB_BOX)
ax.set_xlabel(""); ax.set_xticklabels([short[t.get_text()] for t in ax.get_xticklabels()], fontsize=16)
sns.despine(ax=ax)
name_fig(fig, "12_2b_tool-errors")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=fails, x="harness", y="total_tokens", order=order,
            hue="harness", palette=hpal_b, legend=False, ax=ax,
            showfliers=False, linewidth=1.3, width=0.6)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1e6:.1f}M"))
titled(ax, "Token usage per failed run", SUB_BOX)
ax.set_xlabel(""); ax.set_xticklabels([short[t.get_text()] for t in ax.get_xticklabels()], fontsize=16)
sns.despine(ax=ax)
name_fig(fig, "12_2b_tokens")
plt.tight_layout(); plt.show()

# Success vs failure on the discriminating signals
df.groupby("success")[["tool_error_count", "error_span_count", "total_tokens", "num_llm_calls"]].mean().round(1)


Token usage is **dominated by the benchmark** (`swebench` runs are far
longer than `tau2`), so the pooled boxplot above mostly ranks harnesses by the
suites they ran. (`tool_error_count`, by contrast, is near-zero everywhere except
`appworld` — too sparse to facet usefully.) Holding the benchmark fixed below
isolates each harness's own **token-burn / non-convergence** profile — does a
harness run hot (looping, re-trying) on the *same* tasks?

### 2b-2 · Token usage per failed run, **within each benchmark**

In [ ]:
fb = fails.groupby(["benchmark", "harness"]).size()
keep = fb[fb >= MIN_FAIL].reset_index()
bench_keep2 = sorted(keep.groupby("benchmark").harness.nunique().loc[lambda s: s >= 2].index)

hpal2 = harness_colors(fails.harness.unique())
# One standalone figure per benchmark.
for bench in bench_keep2:
    hs = sorted(keep[keep.benchmark == bench].harness)
    sub = fails[(fails.benchmark == bench) & (fails.harness.isin(hs))]
    fig, ax = plt.subplots(figsize=(max(7, 1.7 * len(hs) + 3), 6))
    sns.boxplot(data=sub, x="harness", y="total_tokens", order=hs,
                hue="harness", palette=hpal2, legend=False, ax=ax,
                showfliers=False, linewidth=1.3, width=0.6)
    ax.set_xlabel(""); ax.set_ylabel("tokens / run")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1e6:.1f}M" if v >= 1e6 else f"{v/1e3:.0f}k"))
    ax.set_xticklabels([f"{h}\n(n={int(fb[(bench, h)])})" for h in hs], fontsize=16)
    titled(ax, f"Token usage per failed run — {bench}",
           f"box = IQR · line = median · whiskers = 1.5×IQR · outliers hidden · ≥{MIN_FAIL} failures/cell")
    sns.despine(ax=ax)
    name_fig(fig, f"13_2b2_{bench}")
    plt.tight_layout(); plt.show()


### 2c · What the judge says — discriminating terms, **on one shared benchmark**

Comparing judge-reasoning vocabulary across harnesses pooled over all suites has the
same confound: each suite has its own vocabulary (`patch`/`test` for swebench,
`flight`/`refund` for tau2). To compare *harnesses* fairly we fix the benchmark to
the one with the broadest harness coverage among failures, and compute each
harness's term frequency relative to the **within-that-benchmark** base — so an
enriched term reflects how that harness fails *on the same tasks*, not which tasks
it ran.

In [ ]:
STOP = set("the a an and or of to in is was for with on at by it that this be as not no "
           "user agent did does provide provided required answer task is were are has have "
           "been which from any clear correct incorrect found evidence based information".split())

def keywords(texts):
    cnt = {}
    for t in texts:
        for w in set(re.findall(r"[a-z]{4,}", t.lower())):
            if w not in STOP:
                cnt[w] = cnt.get(w, 0) + 1
    n = len(texts) or 1
    return {w: c / n for w, c in cnt.items()}

# Pick the benchmark with the most harnesses having >=15 failures (ties -> most failures).
fcount = fails.groupby(["benchmark", "harness"]).size()
cand = (fcount[fcount >= MIN_FAIL].reset_index()
        .groupby("benchmark").agg(nh=("harness", "nunique")))
cand["total"] = fails.groupby("benchmark").size()
FOCUS_BM = cand.sort_values(["nh", "total"], ascending=False).index[0]
fbm = fails[fails.benchmark == FOCUS_BM]
focus_h = sorted(h for h in fbm.harness.unique()
                 if (fcount.get((FOCUS_BM, h), 0) >= MIN_FAIL))

base = keywords(fbm.judge_reasoning)
rows = []
for h in focus_h:
    kw = keywords(fbm[fbm.harness == h].judge_reasoning)
    for w, f in kw.items():
        if f >= 0.10:  # appears in >=10% of this harness's failures on FOCUS_BM
            rows.append({"harness": h, "term": w, "lift": f / (base.get(w, 1e-9))})
lift = pd.DataFrame(rows)
top = (lift[lift.lift > 1].sort_values("lift", ascending=False).groupby("harness").head(6))

# One standalone figure per harness. Bars sorted by lift, so term identity carries no
# extra info — a single brand color avoids rainbow chartjunk.
for h in focus_h:
    th = top[top.harness == h].sort_values("lift")
    if th.empty:
        continue
    fig, ax = plt.subplots(figsize=(9, 0.55 * len(th) + 2))
    ax.barh(range(len(th)), th.lift, color="#3b82f6", height=0.66, zorder=3)
    ax.set_yticks(range(len(th))); ax.set_yticklabels(th.term, fontsize=16)
    for y, v in enumerate(th.lift):
        ax.text(v * 1.02, y, f"{v:.1f}×", va="center", ha="left", fontsize=16,
                weight="bold", color=NEUTRAL["body"])
    ax.axvline(1, ls="--", c=NEUTRAL["muted"], lw=1.3, zorder=2)  # 1 = no enrichment
    ax.set_xlim(0, th.lift.max() * 1.18)
    ax.set_xlabel("lift vs. all failures on this benchmark")
    titled(ax, f"Distinctive judge terms — {h}",
           f"on {FOCUS_BM} only · term appears Nx more than the benchmark's failure base")
    sns.despine(ax=ax, left=True); ax.tick_params(left=False)
    name_fig(fig, f"14_2c_{h}")
    plt.tight_layout(); plt.show()
print(f"focus benchmark: {FOCUS_BM} · harnesses: {focus_h}")


## 3 · Benchmark bleed (contamination) check

Several suites here are **public, static, and pre-date these models' training
cutoffs** — so solutions plausibly leaked into pre-training. Treating a
contaminated score as a measure of *capability* is a mistake; it partly measures
*memorisation*. Risk tiers used below (and easily edited):

| risk | benchmarks | rationale |
|---|---|---|
| **high** | `swebench` | gold patches + issues sit on public GitHub, heavily mirrored & studied |
| **medium** | `tau2_retail`, `tau2_airline`, `tau2_telecom`, `appworld` | public static benchmarks with released task data |
| **low** | `browsecompplus` | live-web retrieval; answers shift, harder to memorise |

The test: **exclude the high-risk suite and see whether the model/harness
ranking and the reliability conclusions from §1 still hold.** If they do, the
findings are robust to bleed; if they move, the leaderboard was partly
measuring recall.

> **Coverage caveat (per §1·0):** the only low-bleed suite, `browsecompplus`, was
> run by **`claude_code` alone**. So the high-vs-clean gap below would silently
> compare SWE-bench-across-many-harnesses against BrowseComp+-on-claude_code — a
> harness confound dressed up as contamination. We therefore compute the gap
> **within `claude_code` only** (the single harness present in both tiers). It
> also means the "clean suites" reliability quadrant in §3c is effectively a
> claude_code view, not a cross-harness one — read it that way.


In [ ]:
RISK = {
    "swebench": "high",
    "tau2_retail": "medium", "tau2_airline": "medium",
    "tau2_telecom": "medium", "appworld": "medium",
    "browsecompplus": "low",
}
df["bleed_risk"] = df.benchmark.map(RISK)
clean = df[df.bleed_risk == "low"]            # contamination-resistant only
no_high = df[df.bleed_risk != "high"]         # drop the worst offender

print("rows by risk tier:")
print(df.bleed_risk.value_counts(), "\n")

# Per-model high-vs-clean gap, restricted to claude_code so the comparison isn't
# confounded by harness (browsecompplus, the only clean suite, is claude_code-only).
cc = df[df.harness == "claude_code"]
gap = []
for m, g in cc.groupby("model"):
    hi = g[g.bleed_risk == "high"]; lo = g[g.bleed_risk == "low"]
    if len(hi) >= 8 and len(lo) >= 8:
        gap.append({"model": m, "high_risk_success": hi.success.mean(),
                    "clean_success": lo.success.mean(),
                    "gap": hi.success.mean() - lo.success.mean(),
                    "n_high": len(hi), "n_low": len(lo)})
gap = pd.DataFrame(gap).sort_values("gap", ascending=False)
print("(gap computed within claude_code only — see coverage caveat above)")
gap.round(3)


### 3a · Contaminated-vs-clean success gap per model

In [ ]:
if len(gap):
    fig, ax = plt.subplots(figsize=(11, 6.5))
    gm = gap.melt(id_vars="model", value_vars=["high_risk_success", "clean_success"],
                  var_name="suite", value_name="rate")
    gm["suite"] = gm["suite"].map({"high_risk_success": "SWE-bench (high bleed)",
                                   "clean_success": "BrowseComp+ (clean)"})
    sns.barplot(data=gm, x="model", y="rate", hue="suite",
                palette=[BRAND["orange"], "#3b82f6"], ax=ax, width=0.7, zorder=3)
    for c in ax.containers:
        ax.bar_label(c, labels=[f"{v*100:.0f}%" for v in c.datavalues],
                     fontsize=16, fontweight="semibold", color=NEUTRAL["body"], padding=3)
    titled(ax, "High-bleed vs. clean success per model",
           "large gap → score may be inflated by memorisation")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("success rate"); ax.set_xlabel("")
    ax.legend(title="", loc="upper right")
    sns.despine(ax=ax)
    name_fig(fig, "15_3a")
    plt.tight_layout(); plt.show()
else:
    print("Not enough overlapping samples to compute per-model bleed gap.")


### 3b · Does the ranking survive excluding the high-risk suite?

> **Coverage gate (per §1·0):** "exclude SWE-bench and see if ranks move" is only
> meaningful for models that *ran* SWE-bench **and** still have ≥2 other suites left
> to rank on afterwards. A model that never ran it can't move, and one that ran
> *only* it has nothing left — including either would fake stability. We restrict to
> the eligible set and **recompute ranks within it**, so the bump chart is a
> like-for-like contamination test, not a coverage artefact.

In [ ]:
def macro_rate(frame, label, min_cell=8):
    cells = rate_table(frame, ["model", "benchmark"])
    cells = cells[cells.n >= min_cell]
    return (cells.groupby("model")
            .agg(**{"rate_" + label: ("success_rate", "mean"),
                    "nbm_" + label: ("benchmark", "nunique")}))

# Eligibility: ran swebench (>=8 tasks) AND >=2 non-swebench suites (>=8) so that
# dropping swebench leaves a comparable, non-empty macro for every model shown.
cmb = rate_table(df, ["model", "benchmark"]); cmb = cmb[cmb.n >= 8]
ran_sb = set(cmb[cmb.benchmark == "swebench"].model)
other_n = cmb[cmb.benchmark != "swebench"].groupby("model").benchmark.nunique()
eligible = sorted(m for m in ran_sb if other_n.get(m, 0) >= 2)

comp = macro_rate(df, "all").join(macro_rate(no_high, "no_high"), how="outer")
comp = comp.loc[[m for m in comp.index if m in eligible]]
# Recompute ranks *within* the eligible set so they're comparable across the two views.
comp = comp.sort_values("rate_all", ascending=False); comp["rank_all"] = range(1, len(comp) + 1)
comp = comp.sort_values("rate_no_high", ascending=False); comp["rank_no_high"] = range(1, len(comp) + 1)
comp["rank_shift"] = comp["rank_no_high"] - comp["rank_all"]
comp = comp.sort_values("rate_all", ascending=False)
print(f"eligible models (ran swebench + >=2 other suites): {eligible}")
display(comp.round(3))

# Slope/bump chart of rank change
fig, ax = plt.subplots(figsize=(10, 6.5))
bump = comp.dropna(subset=["rank_all", "rank_no_high"])
pal = dict(zip(bump.index, [rate_cmap()(v) for v in np.linspace(0.3, 0.9, len(bump))]))
for m, r in bump.iterrows():
    moved = r.rank_no_high != r.rank_all
    ax.plot([0, 1], [r.rank_all, r.rank_no_high], "-o", lw=3 if moved else 1.8,
            color=pal[m], markersize=10, alpha=1 if moved else 0.5, zorder=3)
    ax.text(-0.04, r.rank_all, f"{m}  ({r.rate_all:.0%})", ha="right", va="center",
            fontsize=16, weight="semibold", color=NEUTRAL["body"])
    tag = m if not moved else f"{m}  ▲" if r.rank_no_high < r.rank_all else f"{m}  ▼"
    ax.text(1.04, r.rank_no_high, f"{tag}  ({r.rate_no_high:.0%})", ha="left", va="center",
            fontsize=16, weight="semibold", color=NEUTRAL["body"])
ax.set_xlim(-0.75, 1.75); ax.invert_yaxis()
ax.set_xticks([0, 1]); ax.set_xticklabels(["all benchmarks", "SWE-bench excluded"], fontsize=16)
ax.set_yticks([]); ax.set_ylabel("rank  (1 = best)")
ax.set_title("Model leaderboard — does excluding the high-bleed suite move ranks?", loc="left")
sns.despine(ax=ax, left=True, bottom=True)
name_fig(fig, "16_3b")
plt.tight_layout(); plt.show()


### 3c · Reliability quadrant recomputed on clean suites only

In [ ]:
def quad_panel(frame, title, sub, fname):
    # Benchmark-balanced (macro) x-axis, same as §1b — so the "does it survive?"
    # comparison isn't itself reintroducing the mix confound. Its own standalone image.
    r = config_reliability(frame, min_cell=4, min_total=8).dropna(subset=["macro"])
    r = r[r.n_benchmarks >= 2]  # need >=2 benchmarks for a meaningful cross-task std
    fig, ax = plt.subplots(figsize=(12.5, 8))
    sns.scatterplot(data=r, x="macro", y="cross_task_std", size="n", sizes=(60, 300),
                    hue="macro", palette=rate_cmap(), edgecolor="white",
                    linewidth=1.3, legend=False, ax=ax, zorder=3)
    # Generous right + bottom whitespace so labels for the (often coincident) bottom-right
    # cluster can spread into empty space with clean leader lines. Limits are re-applied
    # after repel (which can nudge them) and the right edge is capped at 100% success.
    xr = max(r.macro.max() - r.macro.min(), 0.05)
    yr = max(r.cross_task_std.max() - r.cross_task_std.min(), 0.05)
    xlim = (r.macro.min() - 0.10 * xr, min(r.macro.max() + 0.55 * xr, 1.02))
    ylim = (r.cross_task_std.min() - 0.45 * yr, r.cross_task_std.max() + 0.18 * yr)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    repel_labels(ax, r.macro, r.cross_task_std, [abbrev_config(c) for c in r.config],
                 expand=(2.0, 2.5), force_text=(1.6, 2.0), force_static=(0.9, 1.2))
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    titled(ax, title, sub)
    ax.set_xlabel("benchmark-balanced success rate")
    ax.set_ylabel("cross-task std of success  →  lower is more predictable")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    sns.despine(ax=ax)
    name_fig(fig, fname)
    plt.tight_layout(); plt.show()
    return r

SUB3C = "x = macro success · y = cross-task std · bubble = sessions · bottom-right = reliable"
quad_panel(df, "Reliability quadrant — all benchmarks", SUB3C, "17_3c_all-benchmarks")
quad_panel(no_high, "Reliability quadrant — SWE-bench excluded", SUB3C, "17_3c_swebench-excluded")


## 4 · Cost — what performance actually costs

These are the **same traces** the [**Open Agent Leaderboard**](https://huggingface.co/blog/ibm-research/open-agent-leaderboard)
(the Exgentic project) scores, so this section is a direct extension of *their* cost view on
*their* data. The leaderboard makes cost a first-class axis: for every configuration it
reports *"the average success rate, the average cost **per task**, and per-benchmark
breakdowns,"* and plots *"every configuration by quality and cost"* to expose the deployable
frontier. Two of its headline cost claims:

1. **"Failed runs cost 20–54% more than successful ones."**
2. **Open-weight models "only tie on cost"** — competitive on price, trailing 18–29 pp on quality.

We **build on this with the data we actually have.** The export carries exact
`total_tokens` per run (the real cost driver) but not a per-run input/output split — so we
price each run with the **same source the paper uses, [LiteLLM's rates](https://github.com/BerriAI/litellm/blob/main/model_prices_and_context_window.json)**
(*"costs are reported using LiteLLM's pricing data"*), blended by each model's
**measured output share** (agents are heavily input-bound — output is ~1–11% of billed
tokens). So the $ here aren't a guess: they're real per-token rates applied to exact token
counts, accurate to a few percent of the exact input+output bill. Then we go past the
leaderboard's headline three ways: a **cost-vs-quality frontier** (their plot, with the
Pareto set marked), **cost per *successful* task** (the number that punishes "cheap because
it gave up"), and a **failed-vs-successful cost split by benchmark** that shows claim #1
*reverses* on conversational suites. Edit `RATE`/`OUT_SHARE` for your own contract rates.

In [ ]:
# Real per-token rates from LiteLLM's pricing table — the SAME source the Open Agent
# Leaderboard / paper use for "cost per task" ("costs are reported using LiteLLM's
# pricing data"). Matched to our deployment strings; $ per 1M tokens, (input, output).
RATE = {
    "claude-opus-4-5": (5.00, 25.00),   # claude-opus-4-5
    "gpt-4.1":         (2.00,  8.00),   # azure/gpt-4.1
    "gpt-5.2":         (1.75, 14.00),   # gpt-5.2
    "gemini-3-pro":    (2.00, 12.00),   # gemini-3-pro-preview
    "kimi-k2.5":       (0.60,  3.00),   # azure_ai/kimi-k2.5
    "deepseek-v3.2":   (0.58,  1.68),   # azure_ai/deepseek-v3.2
}
# We have exact total_tokens per run, but not the per-run input/output split (multi-call
# agents re-send context every call, so summing child-span tokens overcounts). From the
# child llm spans we measured each model's OUTPUT SHARE of billed tokens — agents are
# heavily input-bound — and blend the two rates accordingly. Output is a tiny slice, so a
# blended $/1M-token rate on total_tokens is within a few % of the exact input+output bill.
OUT_SHARE = {  # measured mean output fraction (child spans); edit if you have the exact split
    "claude-opus-4-5": 0.011, "gpt-4.1": 0.109, "gpt-5.2": 0.008,
    "gemini-3-pro":    0.030, "kimi-k2.5": 0.015, "deepseek-v3.2": 0.011,
}
PRICING = {m: RATE[m][0] * (1 - s) + RATE[m][1] * s for m, s in OUT_SHARE.items()}  # blended $/1M
OPEN = {"kimi-k2.5", "deepseek-v3.2"}
df["cost_usd"] = df.total_tokens / 1e6 * df.model.map(PRICING)
print("missing price for:", sorted(set(df.model) - set(PRICING)) or "none")
print("blended $/1M:", {m: round(p, 2) for m, p in PRICING.items()})
df.groupby("model").agg(n=("success", "size"), mean_tok=("total_tokens", "mean"),
                        mean_usd=("cost_usd", "mean"), succ=("success", "mean")).round(2)


### Two cost metrics — read this first

The plots below use two different cost numbers. They are **not** the same, and the gap between
them is the whole point.

| Metric | Definition | What it answers |
|---|---|---|
| **Cost per task** | total \$ spent ÷ **all** runs (wins *and* losses) = mean \$/run | "What does one attempt cost on average?" |
| **Cost per success** | total \$ spent ÷ **successful** runs = (cost per task) ÷ (success rate) | "What do I pay to get one task actually *done* — including paying for the failed attempts?" |

So `cost per success = cost per task ÷ success rate`. A config that succeeds 1-in-6 pays for
~6 attempts per win, so its cost-per-success is ~6× its cost-per-task; a 90%-success config
barely differs between the two. **Neither is per-benchmark** — both pool over every task a
config ran (one number per config). That's fine within a config but mixes suites across
configs, so the headline cost-per-success (4b) inherits the benchmark-mix confound; **4d**
fixes that by recomputing it *within each benchmark*.

### 4a · The cost–quality frontier (their plot, with the Pareto set marked)

This is the Open Agent Leaderboard's headline view — every config placed by **quality**
(benchmark-balanced macro success, y) against **cost** (avg $/task, log x). The dashed line
is the **Pareto frontier**: configs where you can't get more success without paying more.
Anything above/left of another point dominates it. Watch where the **open-weight** configs
land — this is the direct test of *"open only ties on cost."* (Caveat: token cost is heavily
benchmark-driven, so configs that ran the token-heavy coding suites sit further right partly
because of *what they ran* — read alongside §1·0 coverage.)

In [ ]:
relc4 = config_reliability(df, min_cell=5, min_total=20).dropna(subset=["macro"])
cost_cfg = df.groupby("config").agg(usd_task=("cost_usd", "mean"),
                                    tok_task=("total_tokens", "mean"),
                                    model=("model", "first")).reset_index()
P = relc4.merge(cost_cfg, on="config")
P["open"] = P.model.isin(OPEN)
# Pareto frontier: walk configs cheapest-first, keep each new success high-water mark.
front, best = [], -1.0
for _, r in P.sort_values("usd_task").iterrows():
    if r.macro > best + 1e-9:
        front.append(r.config); best = r.macro

fig, ax = plt.subplots(figsize=(13, 8))
for isopen, grp in P.groupby("open"):
    ax.scatter(grp.usd_task, grp.macro, s=80 + grp.n * 0.7,
               color=BRAND["orange"] if isopen else "#3b82f6", alpha=0.85,
               edgecolor="white", linewidth=1.4, zorder=3,
               label="open-weight" if isopen else "closed")
fr = P[P.config.isin(front)].sort_values("usd_task")
ax.plot(fr.usd_task, fr.macro, "--o", color=BRAND["violet"], lw=2, ms=7, zorder=2,
        label="cost–quality frontier")
repel_labels(ax, P.usd_task, P.macro, [abbrev_config(c) for c in P.config],
             expand=(1.5, 1.9), force_text=(0.9, 1.3), force_static=(0.4, 0.7))
ax.set_xscale("log")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.2f}"))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel("avg cost per task  →  cheaper is better  (log $, LiteLLM rates)")
ax.set_ylabel("benchmark-balanced (macro) success  →  higher is better")
titled(ax, "Cost vs quality — the deployable frontier",
       "upper-left dominates · bubble = sessions · $ = LiteLLM rates × measured output share", pad=20)
ax.legend(loc="upper left", frameon=True, framealpha=0.9, edgecolor="#cccccc")
ax.margins(0.12); sns.despine(ax=ax)
name_fig(fig, "18_4a")
plt.tight_layout(); plt.show()
P.sort_values("macro", ascending=False)[
    ["config", "n", "macro", "usd_task", "tok_task"]].round({"macro": 3, "usd_task": 2}).head(12)


### 4b · Cost per *successful* task — where "cheap" stops being cheap

Average cost per task flatters configs that fail a lot: a run that bails early is cheap, but
you paid for nothing. The honest denominator is **successes**, not attempts —
`total spend ÷ number of solved tasks`. A config that succeeds 16% of the time effectively
pays for ~6 attempts per win. This is the §8 slogan — *"efficiency without success is just
failure with fewer tokens"* — turned into one number, and the rank order is very different
from raw $/task.

In [ ]:
rows = []
for c, sub in df.groupby("config"):
    n, ns = len(sub), int(sub.success.sum())
    if n >= 20 and ns > 0:
        rows.append({"config": c, "n": n, "succ": sub.success.mean(),
                     "usd_success": sub.cost_usd.sum() / ns,
                     "tok_success": sub.total_tokens.sum() / ns})
cps = pd.DataFrame(rows).sort_values("usd_success").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12.5, 0.56 * len(cps) + 2))
cmap = rate_cmap()
ax.barh(range(len(cps)), cps.usd_success, color=[cmap(s) for s in cps.succ],
        height=0.72, zorder=3, edgecolor="white")
ax.set_yticks(range(len(cps))); ax.set_yticklabels(cps.config, fontsize=16)
ax.set_xscale("log")
for y, (_, r) in enumerate(cps.iterrows()):
    ax.text(r.usd_success * 1.06, y, f"${r.usd_success:,.2f}  ·  {r.succ:.0%} succ",
            va="center", ha="left", fontsize=16, weight="semibold", color=NEUTRAL["body"])
ax.invert_yaxis()  # cheapest-per-success on top
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.set_xlim(right=cps.usd_success.max() * 2.2)
ax.set_xlabel("cost per SUCCESSFUL task  (log $, LiteLLM rates) — total spend ÷ successes")
titled(ax, "Cost per successful outcome — the cheap-but-failing configs are the priciest",
       "bar colour = success rate (pale → deep blue) · $ = LiteLLM rates × measured output share", pad=20)
sns.despine(ax=ax, left=True); ax.tick_params(left=False)
name_fig(fig, "19_4b")
plt.tight_layout(); plt.show()
cps.assign(usd_per_success=cps.usd_success.round(2), tok_per_success=cps.tok_success.round(0))[
    ["config", "n", "succ", "usd_per_success", "tok_per_success"]].round({"succ": 3})


### 4c · Do failures really cost more? Only on coding tasks

The leaderboard's *"failed runs cost 20–54% more"* is true **pooled** (ours is even higher,
+77%) — but pooling hides a clean reversal. Holding the benchmark fixed, **coding/agentic
failures cost more** (the thrash pattern from §8 — the agent loops and burns tokens before
failing) while **conversational `tau2` failures cost *less*** (the give-up pattern — the agent
bails early and cheaply). So a blanket "failures are expensive" guardrail is exactly backwards
for half the task families. Ratio = mean tokens of failed runs ÷ mean tokens of successful
runs; the yellow band marks the leaderboard's pooled +20–54%.

In [ ]:
rows = []
for b, sub in df.groupby("benchmark"):
    s = sub[sub.success == 1].total_tokens.mean()
    f = sub[sub.success == 0].total_tokens.mean()
    if pd.notna(s) and pd.notna(f) and s > 0:
        rows.append({"benchmark": b, "ratio": f / s, "n_fail": int((sub.success == 0).sum())})
rc = pd.DataFrame(rows).sort_values("ratio").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 5.8))
colors = [BRAND["orange"] if r > 1 else "#3b82f6" for r in rc.ratio]
ax.bar(range(len(rc)), rc.ratio, color=colors, width=0.62, zorder=3)
ax.axhline(1.0, color=NEUTRAL["muted"], lw=1.2, zorder=2)                # parity
ax.axhspan(1.20, 1.54, color="#eab308", alpha=0.16, zorder=0)           # leaderboard's band
ax.text(len(rc) - 0.5, 1.37, "Open Agent Leaderboard\npooled +20–54%", ha="right",
        va="center", fontsize=16, color="#a16207")
for x, r in enumerate(rc.ratio):
    ax.text(x, r + 0.04, f"{r:.2f}×\n({(r - 1) * 100:+.0f}%)", ha="center", va="bottom",
            fontsize=16, weight="bold", color=NEUTRAL["body"])
ax.set_xticks(range(len(rc)))
ax.set_xticklabels([BENCH_ABBR.get(b, b) for b in rc.benchmark], rotation=0, ha="center")
ax.set_ylabel("failed-run tokens ÷ successful-run tokens")
ax.set_ylim(0, max(rc.ratio) * 1.18)
titled(ax, "Do failures cost more? Only on coding tasks",
       "orange >1 = failures cost MORE (thrash) · blue <1 = failures cost LESS (give up) · "
       "yellow = leaderboard's pooled claim", pad=20)
sns.despine(ax=ax)
name_fig(fig, "20_4c")
plt.tight_layout(); plt.show()


### 4d · Cost per success, **within each benchmark** (apples-to-apples)

4b's cost-per-success pools over whatever suites a config ran, so a config that mostly ran the
cheap `tau2` tasks looks cheaper than one that ran token-heavy `swebench` — independent of the
harness. This holds the **benchmark fixed**: each panel is one suite, and within it every
config faces the same task type, so `total $ ÷ successes` is now a clean comparison. We split
the suites into the **coding / agentic** family (swebench, appworld, browse — high token
volume) and the **conversational τ²** family (airline, retail, telecom), since they live on
very different cost scales. This is the honest test of "does open *actually* cost less per
solved task," and of which harness wins once you're paying per outcome on a given workload.

In [ ]:
CODING = ["swebench", "appworld", "browsecompplus"]
CONV   = ["tau2_airline", "tau2_retail", "tau2_telecom"]
rows = []
for (b, c), sub in df.groupby(["benchmark", "config"]):
    n, ns = len(sub), int(sub.success.sum())
    if n >= 10 and ns > 0:                       # enough runs on this suite, and ≥1 success
        rows.append({"benchmark": b, "config": c, "succ": sub.success.mean(),
                     "usd_success": sub.cost_usd.sum() / ns})
cb = pd.DataFrame(rows)
cmap = rate_cmap()

SUB = ("suite held fixed · bar colour = success rate (pale → deep blue) · "
       "$ = LiteLLM rates × measured output share · configs with ≥10 runs · cheapest on top")

# One standalone figure per benchmark (no more stacked panels). `fam_rank` orders the
# files so the coding suites come before the conversational ones in the export.
ORDER = [(b, "coding") for b in CODING] + [(b, "conv") for b in CONV]
for rank, (b, fam) in enumerate((b, fam) for b, fam in ORDER if b in set(cb.benchmark)):
    s = cb[cb.benchmark == b].sort_values("usd_success").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(11, 0.6 * len(s) + 2))
    ax.barh(range(len(s)), s.usd_success, color=[cmap(v) for v in s.succ],
            height=0.72, zorder=3, edgecolor="white")
    ax.set_yticks(range(len(s))); ax.set_yticklabels(s.config, fontsize=16)
    ax.set_xscale("log")
    for y, (_, r) in enumerate(s.iterrows()):
        ax.text(r.usd_success * 1.10, y, f"${r.usd_success:,.2f}  ·  {r.succ:.0%}",
                va="center", ha="left", fontsize=16, weight="semibold", color=NEUTRAL["body"])
    ax.invert_yaxis()                              # cheapest-per-success on top
    ax.set_xlim(right=s.usd_success.max() * 3.4)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda v, _: f"${v:,.2f}" if v < 10 else f"${v:,.0f}"))
    ax.set_xlabel("cost per SUCCESSFUL task  (log $)")
    titled(ax, f"Cost per successful task — {b}", SUB)
    sns.despine(ax=ax, left=True); ax.tick_params(left=False)
    name_fig(fig, f"21_4d_{rank+1:02d}_{b}")
    plt.tight_layout(); plt.show()


**Cost takeaways (building on the Open Agent Leaderboard).**

- **Pick on cost *per success*, not cost per task.** The cheapest-per-task configs (minimal
  scaffolds that give up) are often the *most* expensive per solved task. The frontier in 4a
  is the deployable view; 4b is the one that survives contact with a real workload.
- **"Open only ties on cost" undersells it at these rates.** Open models burn *more tokens*
  (4b, token column) but cost ~8–9× less per token (LiteLLM: ~$0.6/M vs Opus ~$5.2/M), so on
  $/task they don't just tie — the reliable open `claude_code` configs are among the cheapest
  *per success*. This stays rate-dependent: the gap narrows or flips at different contract
  rates, so re-run with your own `RATE`.
- **"Failures cost more" is a coding-suite rule, not a universal one (4c).** Guardrails that
  cap spend on "abnormally expensive" runs catch coding thrash but miss conversational give-up,
  where the *cheap, short* run is the failure. Budget alarms need per-task-family thresholds.

Knobs: the `RATE` / `OUT_SHARE` tables (pricing), the `min_total=20` cutoff, and the `OPEN` set.

## 5 · How to use this

**The coverage caveat runs through everything.** Only 2 of 22 configs ran all six
suites (§1·0); `browsecompplus` is `claude_code`-only. So *any* number pooled across
benchmarks is partly a statement about which suites a config happened to run. Every
cross-config / cross-harness claim below is the version that equalises coverage —
either by restricting to a shared suite set, faceting by benchmark, or adjusting for
benchmark as a covariate.

- **§1 — reliability.** Rank by `floor` (Wilson lower bound), not raw average, and
  ignore the grey (<3-suite) bars for cross-config comparison. The harness-effect
  headline survives the coverage fix: in the adjusted model (§1h) `claude_code` is
  **+28 pp over `tool_calling` holding model and benchmark fixed**, and harness still
  adds more incremental R² than model (≈5% vs ≈1%) — but the *raw* §1d gap was
  inflated by coverage, and some per-model swings (e.g. claude-opus in §1e) shrink to
  near-noise once you restrict to shared suites. Believe §1g/§1h over §1a/§1d.
- **§2 — failure modes.** Read the **benchmark-faceted** panels (§2a-2, §2b-2, §2c),
  not the pooled ones: a harness "fingerprint" that changes suite-to-suite was a
  coverage artefact; one that holds within every benchmark is a real harness
  signature worth designing mitigations around.
- **§3 — bleed.** The contamination tests are gated to comparable coverage: the
  high-vs-clean gap is computed within `claude_code` only, and the rank-survival test
  only to models that ran SWE-bench plus ≥2 other suites. Re-run with your own `RISK`
  mapping to match your training-cutoff knowledge.
- **§4 — cost.** This is the same dataset the Open Agent Leaderboard scores, so it
  directly extends their cost view — and prices runs with **LiteLLM's rates, the same
  source the paper uses**, blended by each model's measured output share. Tokens are exact;
  $ are accurate to a few percent (edit `RATE`/`OUT_SHARE` for your contract). Choose
  configs on **cost per success** (§4b), not cost per task, and remember the
  "failures cost more" rule is coding-only (§4c).

Knobs to tweak: `MIN_N`, `COMPARABLE_BM`, `MIN_FAIL`, the `classify()` thresholds,
the keyword `STOP` set, the `RISK` dictionary, and the `PRICING` table.
